# Data Analysis and Processing Notebook

Project : Decolonizing Health Professions Education Research: An Analysis of Global Network Patterns and Equity Implications

Author : P. Sitthirat et al.

This notebook serves as a comprehensive guide for processing and analyzing the network of co-authorship in health professions education community. It includes data cleaning, manipulation, and analysis.

Objectives:
- Processing the bibliometric database.
- Geocoding for the author.
- Conducting social network analysis.

Requirements: Ensure the following libraries are installed before running the notebook: `pip install -r scripts/requirements.txt`

### 1. Initial Setup and Library Imports

This cell initializes the notebook with necessary configurations and imports key libraries and custom modules for data processing, geocoding, and network analysis.

### Features:
1. **IPython Extensions**:
   - `%load_ext autoreload`: Automatically reloads modules before executing cells, ensuring the latest code changes are applied.
   - `%autoreload 2`: Reloads all imported modules every time a cell is executed.

2. **General Library Imports**:
   - `os`: For interacting with the operating system.
   - `pandas`: For data manipulation and analysis.
   - `numpy`: For numerical operations.
   - `tqdm`: For progress bars in loops.
   - `geopandas`: For geographical data manipulation and analysis.
   - `matplotlib.pyplot`: For main visualization.

3. **Advanced Library Imports**:
   - `itertools.zip_longest`: Allows iteration over multiple lists of unequal length, filling missing values with a specified placeholder so no data is lost.
   - `fuzzywuzzy`: Fuzzy string matching, which compares two text strings and returns a similarity score (0–100).
   - `inset_axes`: For zooming some region in the big map.
   - `upsetplot`: For building the upset plot.
   - `networkx`: For social network analysis (base libraries, please use with `scripts/network.py`).
   - `community`: Louvain community detection (python-louvain), used for modularity and participation-coefficient analysis.

3. **Custom Modules** from `scripts` folder:
   - `geocoder`: For geocoding the affiliation dataset, including the batch `geocode_affiliations` pipeline.
   - `network`: For building co-authorship graphs and calculating network parameters.
   - `visualization`: For the choropleth, bump chart, and network-on-map plotting functions reused throughout Section 3.
   - `process.DataManipulation`: For parsing author/affiliation strings and imputing missing geocoding results.

4. **Notebook Settings**:
   - `clear_output`: For clearing the output before further operations
   - `pd.set_option('future.no_silent_downcasting', True)`: Ensures that pandas raises warnings for downcasting operations.
   - `warnings.simplefilter('ignore', DeprecationWarning)`: Suppresses deprecation warnings for a cleaner output.

### Code:
This configuration ensures an efficient and clean working environment for data analysis and processing.

In [ ]:
%load_ext autoreload
%autoreload 2

# General libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
# from tqdm import tqdm
import geopandas as gpd
import matplotlib.pyplot as plt

# # Advanced libraries
# from itertools import zip_longest
# from fuzzywuzzy import fuzz
# from fuzzywuzzy import process
# from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from upsetplot import UpSet, from_indicators
import networkx as nx
import community as community_louvain

# # Custom modules
from scripts import scrp_article
from scripts import network
from scripts import visualization
# from scripts.process import DataManipulation

from IPython.display import clear_output
# pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)

#### Global Map and Official Language Imports

Due to we have to analyze and visualize the data into the global map, `world` would be imported to be the base map for further analysis
Reference map was downloaded from [Natural Earth Data](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/) which is 10-metre resolution of country border.

Morover, official language of each countries was extracted from [Resource Watch](https://resourcewatch.org/data/explore/soc_071_world_languages?section=Discover&selectedCollection=&zoom=3&lat=0&lng=0&pitch=0&bearing=0&basemap=dark&labels=light&layers=%255B%257B%2522dataset%2522%253A%252220662342-dcdd-4a42-9f58-bcc80217de71%2522%252C%2522opacity%2522%253A1%252C%2522layer%2522%253A%2522f2d76e6b-060d-4dc9-83ea-284bef6b2aae%2522%257D%255D&aoi=&page=1&sort=most-viewed&sortDirection=-1).

These two files would be first imported using `geopandas`.

In [ ]:
world = gpd.read_file('data/world-map/ne_10m_admin_0_countries.zip')
world = world[world['ADMIN'] != 'Antarctica']

language = gpd.read_file('data/world-map/world_languages.zip')

world = world.merge(language[['COUNTRY', 'FIRST_OFFI']], how='inner', left_on='GEOUNIT', right_on='COUNTRY')

#### Setting the Visualization

For consistent visulization all visual parameters such as font, size, color would be set here.

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Arial"],
    "axes.edgecolor": "black",
    "axes.linewidth": 1,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

### 2. Data Retrieval and Management

This section imports the bibliometric database from [OpenAlex API](https://openalex.org/) which are the scholar outputs from five journals between Jan 2015 to June 2026 period.

In [ ]:
journal = pd.read_csv('data/source_id.csv')
group = 'meded' # changeable
start_date = "2015-01-01"
end_date = "2026-06-30"

works_scraped = pd.DataFrame()
authorships_scraped = pd.DataFrame()

for id in journal[journal['group'] == group]['id']:

    works = scrp_article.retrieve_journal_works(
        source_id=id,
        start_date=start_date,
        end_date=end_date,
    )

    w_df, a_df = scrp_article.flatten_works(works)

    works_scraped = pd.concat([works_scraped, w_df], ignore_index=True)
    authorships_scraped = pd.concat([authorships_scraped, a_df], ignore_index=True)

    clear_output()

scraped_path = Path(f"output/scraped/{group}")
scraped_path.mkdir(parents=True, exist_ok=True)
works_scraped.to_csv(scraped_path / 'works_scraped.csv', index=True)
authorships_scraped.to_csv(scraped_path / 'authorships_scraped.csv', index=True)

print(f"Retrieval successful from {len(journal[journal['group'] == group]['id'])} journals from {start_date} to {end_date} including {len(works_scraped)} initial works. Scraped files were saved in 'output/scraped/{group}' folder.")

In [ ]:
works_df = pd.read_csv(scraped_path / 'works_scraped.csv')

# Include only some type of articles
included_work_types = ['article', 'review', 'book-chapter', 'editorial', 'conference-abstract']
works_df = works_df[works_df['work_type'].isin(included_work_types)].copy()

# Export valid work dataframe
metadata_path = Path(f"data/metadata/{group}")
metadata_path.mkdir(parents=True, exist_ok=True)
works_df.to_csv(metadata_path / 'works.csv', index=False)

In [ ]:
from scripts import geocode_workflow

authorships_df = pd.read_csv(scraped_path / 'authorships_scraped.csv')

# Include only author of valid works and drop duplicated rows
authorships_df = authorships_df[authorships_df['work_id'].isin(works_df['work_id'])].copy()
authorships_df = authorships_df.drop_duplicates(subset=['work_id', 'author_id']).reset_index(drop=True)

# Fills institution_country_code from the shared crosswalks in output/map/
# (institution_id -> affiliation -> author_id) plus same-author/same-work
# inference. See scripts/geocode_workflow.py.
authorships_df = geocode_workflow.impute_country_codes(authorships_df)

# If rows are still missing a country code afterwards, export the
# not-yet-labeled institutions/affiliations for this field, fill in the
# institution_country_code column (e.g. by asking Claude to label them, same
# as this session), then merge the labels back into the shared crosswalk:
#
#   geocode_workflow.export_pending(authorships_df, 'institution', group)
#   geocode_workflow.export_pending(authorships_df, 'affiliation', group)
#   # ...label output/map/pending/geocode_{institution,affiliation}_{group}.csv...
#   geocode_workflow.merge_labels('institution', f'output/map/pending/geocode_institution_{group}.csv')
#   geocode_workflow.merge_labels('affiliation', f'output/map/pending/geocode_affiliation_{group}.csv')
#
# ...then re-run this cell to pick up the newly-labeled rows.

In [ ]:
authorships_df.to_csv(metadata_path / 'authorships.csv', index=False)

### 3. Country-level Analysis

We analysed the distribution and network pattern using the retrievel metadata dataframe from `data/metadata` folder. The analysis included:
- display distribution of scholar output in each country and income level
- social network analysis

In [ ]:
# Import cleaned publication and country group datset
works_df = pd.read_csv(metadata_path / 'works.csv')
authorships_df = pd.read_csv(metadata_path / 'authorships.csv')

income_gr = pd.read_csv('data/world-map/country_group.csv')
income_gr = income_gr.merge(world[['ADM0_A3', 'FIRST_OFFI', 'ISO_A2_EH']], how='left', left_on='Code', right_on='ADM0_A3')

# 15 World Bank economies have no Natural Earth match on ADM0_A3 and come back
# with ISO_A2_EH = NaN. pandas joins NaN to NaN, so without this every authorship
# row that has no country code matches all 15 and is duplicated with a spurious
# income group. Patch the codes we can recover, then drop the rest before merging.
iso2_patch = {
    'BHS': 'BS', 'CHI': 'JE', 'COG': 'CG', 'CPV': 'CV', 'CUW': 'CW',
    'CZE': 'CZ', 'GNB': 'GW', 'MAC': 'MO', 'MKD': 'MK', 'SSD': 'SS',
    'STP': 'ST', 'SWZ': 'SZ', 'TZA': 'TZ', 'VIR': 'VI', 'XKX': 'XK',
}
income_gr['ISO_A2_EH'] = income_gr['ISO_A2_EH'].fillna(income_gr['Code'].map(iso2_patch))
income_gr = income_gr.dropna(subset=['ISO_A2_EH']).drop_duplicates(subset='ISO_A2_EH')

df_foranalyse = authorships_df.merge(income_gr[['ISO_A2_EH', 'Income group', 'Region']], how='left', left_on='institution_country_code', right_on='ISO_A2_EH')
df_foranalyse = df_foranalyse.merge(world[['ISO_A2_EH', 'FIRST_OFFI']].drop_duplicates(subset='ISO_A2_EH'), how='left', on='ISO_A2_EH')

# The merges are lookups, not joins -- they must never change the row count.
assert len(df_foranalyse) == len(authorships_df), 'income/language merge duplicated authorship rows'

display(df_foranalyse)

#### Publication trends by income group and country

In [ ]:
# Count publications per income group
counts_income = df_foranalyse.groupby(['publication_year', 'Income group'])['work_id'].nunique().reset_index()
counts_income_pivot = counts_income.pivot(index='publication_year', columns='Income group', values='work_id').fillna(0)
display(counts_income_pivot)

plt.figure(figsize=(10, 6))

for col in counts_income_pivot.columns:
    plt.plot(counts_income_pivot.index, counts_income_pivot[col], marker='o', label=col)

plt.title("Unique Work ID Counts per Income Group by Year")
plt.xlabel("Year")
plt.ylabel("Number of Unique DOIs")
plt.legend(title="Income Group")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Count authors per income group
counts_author = df_foranalyse.groupby(['publication_year', 'Income group'])['author_id'].nunique().reset_index()
counts_author_pivot = counts_author.pivot(index='publication_year', columns='Income group', values='author_id').fillna(0)
display(counts_author_pivot)

plt.figure(figsize=(10, 6))

for col in counts_author_pivot.columns:
    plt.plot(counts_author_pivot.index, counts_author_pivot[col], marker='o', label=col)

plt.title("Unique Author ID Counts per Income Group by Year")
plt.xlabel("Year")
plt.ylabel("Number of Unique Author")
plt.legend(title="Income Group")
plt.grid(True)
plt.tight_layout()
plt.show()

#### Global and regional publication distribution

In [ ]:
# Count publications per country
counts_country = df_foranalyse.groupby(['Income group', 'institution_country_code'])['work_id'].nunique().reset_index()
counts_country.columns = ['Income group', 'ISO_A2_EH', 'Publication Count']

# Merge author data with map data
world_plt = world[world['ADMIN'] != 'Antarctica']  # Exclude Antarctica
merged = world_plt.merge(counts_country, on='ISO_A2_EH', how='left')
merged['Publication Count'] = merged['Publication Count'].fillna(0)
merged['Log Publication Count'] = np.log1p(merged['Publication Count'])

# Separate subsets for plotting
has_published = merged[merged['Publication Count'] > 0]
no_published = merged[merged['Publication Count'] == 0]

visualization.plot_choropleth(world_plt, has_published, no_published)

In [ ]:
# Plot some region
visualization.plot_choropleth(world_plt, has_published, no_published, xlim=(-100, -60), ylim=(10, 30), title="Caribbean Region")
visualization.plot_choropleth(world_plt, has_published, no_published, xlim=(-25, 45), ylim=(34, 72), title="Europe Region")
visualization.plot_choropleth(world_plt, has_published, no_published, xlim=(90, 135), ylim=(-10, 20), title="Southeast Asia Region")

#### Collaboration patterns across income groups

In [ ]:
# Pivot the collaboration patterns
df_pivot = df_foranalyse.groupby(by=['work_id', 'publication_year']).agg({
    'institution_country_code': set,
    'Region': set,
    'Income group': set
}).reset_index()

df_pivot['no_Country'] = df_pivot['institution_country_code'].apply(lambda x: len(x))
df_pivot['no_Income'] = df_pivot['Income group'].apply(lambda x: len(x))

df_pivot['count_grp'] = df_pivot.apply(
    lambda row: (
        'High only' if (row['no_Income'] == 1 and 'High income' in str(row['Income group'])) else
        'Developing only' if ('High income' not in str(row['Income group'])) else
        'Mixed'
    ),
    axis=1
)

pivot_counts = df_pivot.pivot_table(
    index='publication_year',
    columns='count_grp',
    aggfunc='size',
    fill_value=0
).reset_index()

display(pivot_counts)

#### Multi-income-group collaboration (UpSet plot)

In [ ]:
# Create upset plot
df_pivot = df_foranalyse.groupby(by=['work_id', 'publication_year']).agg({
    'institution_country_code': set,
    'Region': set,
    'Income group': set
}).reset_index()
df_pivot['no_Country'] = df_pivot['institution_country_code'].apply(lambda x: len(x))
subset = df_pivot[df_pivot['no_Country'] > 1]['work_id'].drop_duplicates()
subset = pd.DataFrame(subset, columns=['work_id'])

high_income = df_foranalyse[df_foranalyse['Income group'] == 'High income']['work_id'].copy()
upper_middle_income = df_foranalyse[df_foranalyse['Income group'] == 'Upper middle income']['work_id'].copy()
lower_middle_income = df_foranalyse[df_foranalyse['Income group'] == 'Lower middle income']['work_id'].copy()
low_income = df_foranalyse[df_foranalyse['Income group'] == 'Low income']['work_id'].copy()

# Add membership flags
subset['High income'] = subset['work_id'].isin(high_income)
subset['Upper middle income'] = subset['work_id'].isin(upper_middle_income)
subset['Lower middle income'] = subset['work_id'].isin(lower_middle_income)
subset['Low income'] = subset['work_id'].isin(low_income)

upset_data = from_indicators(['High income', 'Upper middle income', 'Lower middle income', 'Low income'], subset)
display(upset_data)
# Plot
# UpSet(upset_data, subset_size='count', show_counts=True).plot()
# plt.show()

#### Global co-authorship network construction (2015–2024)

Build a yearly co-authorship graph per country and accumulate it into a single running network `G_all`, tracking centrality and network statistics (via `scripts/network.py`) as each year is added.

In [ ]:
# Social Network Analysis
results = pd.DataFrame()
G_all = nx.Graph()

for year in range(2015, 2026):
    # Filter only the data for the current year
    df_year = df_foranalyse[df_foranalyse['publication_year'] == year]

    # Use the existing function to construct the graph for this year
    G_year = network.network_coauthorship(df_year, 'institution_country_code', id_col='work_id', node_label=['Region', 'Income group', 'FIRST_OFFI'])

    # Merge G_year into G_all
    for u, v, data in G_year.edges(data=True):
        if G_all.has_edge(u, v):
            G_all[u][v]['weight'] += data['weight']
        else:
            G_all.add_edge(u, v, weight=data['weight'])

    for node, data in G_year.nodes(data=True):
        if node not in G_all:
            G_all.add_node(node)
        for attr, value in data.items():
            G_all.nodes[node][attr] = value

    result = network.network_params(df_year, G_all, homophily_attr=['Region', 'Income group', 'FIRST_OFFI'])
    result['publication_year'] = year
    results = pd.concat([results, result], ignore_index=True)

    # Degree centrality
    deg_cent = nx.degree_centrality(G_all)
    top_deg = sorted(deg_cent.items(), key=lambda x: x[1], reverse=True)[:10]

    print("Top 10 Degree Centrality:")
    for node, val in top_deg:
        print(f"{node}: {val:.4f}")

    # Betweenness centrality
    bet_cent = nx.betweenness_centrality(G_all, normalized=True)
    top_bet = sorted(bet_cent.items(), key=lambda x: x[1], reverse=True)[:10]

    print("\nTop 10 Betweenness Centrality:")
    for node, val in top_bet:
        print(f"{node}: {val:.4f}")

    print(f'Year {year} network analysed.')

G_country = G_all
display(results)

#### Network snapshots and centrality distributions

Build the two temporal network snapshots (2015–2019 and cumulative through 2024) used by the bump charts below, and inspect the distribution of centrality/participation scores across the full network.

In [ ]:
# Define color map for income categories
income_colors = {
    'High income': '#1b9e77',
    'Upper middle income': '#d95f02',
    'Lower middle income': '#7570b3',
    'Low income': '#e7298a',
    'Other': '#ffffff'
}

# Keep top N countries per year for the bump charts below
top_n = 15

G_2020 = network.network_coauthorship(df_foranalyse, 'institution_country_code', id_col='work_id', filter_col='publication_year', filter_value=range(2015, 2020))
G_2024 = network.network_coauthorship(df_foranalyse, 'institution_country_code', id_col='work_id')

In [ ]:
G = network.network_coauthorship(df_foranalyse, 'institution_country_code', id_col='work_id')

deg = nx.degree_centrality(G)
cls = nx.closeness_centrality(G)
btw = nx.betweenness_centrality(G)

partition = community_louvain.best_partition(G, random_state=42)
pc = network.participation_coefficient(G, partition)

# Convert to lists
deg_values = list(deg.values())
cls_values = list(cls.values())
btw_values = list(btw.values())
part_values = list(pc.values())

# Plotting
fig, axs = plt.subplots(4, 1, figsize=(8, 16), sharex=False)
bins = np.linspace(0, 1, 31)  # 30 equal-width bins between 0 and 1

# Degree centrality
axs[0].hist(deg_values, bins=bins, color='grey', edgecolor='black')
axs[0].set_title('Degree Centrality', fontsize=12)
axs[0].set_ylabel('Frequency')
axs[0].set_xlim(0,1)
axs[0].tick_params(labelbottom=False)  # Hide x-axis labels

# Closeness centrality
axs[1].hist(cls_values, bins=bins, color='grey', edgecolor='black')
axs[1].set_title('Closeness Centrality', fontsize=12)
axs[1].set_ylabel('Frequency')
axs[1].set_xlim(0,1)
axs[1].tick_params(labelbottom=False)  # Hide x-axis labels

# Betweenness centrality
axs[2].hist(btw_values, bins=bins, color='grey', edgecolor='black')
axs[2].set_title('Betweenness Centrality', fontsize=12)
axs[2].set_ylabel('Frequency')
axs[2].set_xlim(0,1)
axs[2].tick_params(labelbottom=False)  # Hide x-axis labels

# Participation coefficient
axs[3].hist(part_values, bins=bins, color='grey', edgecolor='black')
axs[3].set_title('Participation Coefficient', fontsize=12)
axs[3].set_ylabel('Frequency')
axs[3].set_xlabel('Centrality Score')
axs[3].set_xlim(0,1)

# Tight layout
plt.tight_layout()
plt.show()

#### Bump charts: centrality rank shifts, 2015 vs 2024

For each centrality measure, rank the top countries in the 2015–2019 snapshot (`G_2020`) and the cumulative 2024 snapshot (`G_2024`), and plot how their rank shifted using `visualization.plot_bump_chart`.

In [ ]:
# Degree centrality
deg_2015 = nx.degree_centrality(G_2020)
deg_2024 = nx.degree_centrality(G_2024)
visualization.plot_bump_chart(deg_2015, deg_2024, income_gr, income_colors, top_n=10)

In [ ]:
# Closeness centrality
cls_2015 = nx.closeness_centrality(G_2020)
cls_2024 = nx.closeness_centrality(G_2024)
visualization.plot_bump_chart(cls_2015, cls_2024, income_gr, income_colors, top_n=top_n)

In [ ]:
# Betweenness centrality
btw_2015 = nx.betweenness_centrality(G_2020)
btw_2024 = nx.betweenness_centrality(G_2024)
visualization.plot_bump_chart(btw_2015, btw_2024, income_gr, income_colors, top_n=top_n)

In [ ]:
# Participation coefficient
partition_2015 = community_louvain.best_partition(G_2020, random_state=42)
partition_2024 = community_louvain.best_partition(G_2024, random_state=42)

pc_2015 = network.participation_coefficient(G_2020, partition_2015)
pc_2024 = network.participation_coefficient(G_2024, partition_2024)

visualization.plot_bump_chart(pc_2015, pc_2024, income_gr, income_colors, top_n=top_n)

#### Co-authorship network on the world map

Overlay the co-authorship network on the world map (via `visualization.plot_network_on_map`), first for all publications, then restricted to publications with a low-income first author.

In [ ]:
df = df_foranalyse.copy()

G_lic = network.network_coauthorship(df, 'institution_country_code', id_col='work_id')
country_income = df[['institution_country_code', 'Income group']].drop_duplicates().set_index('institution_country_code')['Income group'].to_dict()
for node in G_lic.nodes:
    G_lic.nodes[node]['Income group'] = country_income.get(node, 'Other')  # default to 'Other' if missing

visualization.plot_network_on_map(
    G_lic, world, income_colors, edge_scale=0.05,
    title='Co-authorship Network on World Map\nNode Color = Income Group, Labels = LICs Only'
)

In [ ]:
df = df_foranalyse.copy()
lic_set = set(df[(df['Income group'] == 'Low income') & (df['author_position'] == 'first')]['work_id'])
df_lic = df[df['work_id'].isin(lic_set)]

G_lic = network.network_coauthorship(df_lic, 'institution_country_code', id_col='work_id')
country_income = df_lic[['institution_country_code', 'Income group']].drop_duplicates().set_index('institution_country_code')['Income group'].to_dict()
for node in G_lic.nodes:
    G_lic.nodes[node]['Income group'] = country_income.get(node, 'Other')  # default to 'Other' if missing

visualization.plot_network_on_map(
    G_lic, world, income_colors, edge_scale=0.5, node_scale=10,
    title='Co-authorship Network on World Map\nNode Color = Income Group, Labels = LICs Only'
)